In [3]:
# # 강화학습 5대 중요 키워드
# # 정책을 세우는것이 학습의 최종 목표

# class ReinforcementLearningDemo():
#     def __init__(self):
#         self.score = 0
#         self.game_over = False

#     def cartpole_example(self):
#         print("목표 : 막대 쓰러지지 않기")
#         while not self.game_over:
#             state = {"pole_angle" : 0.1,
#                      "cart_position" : 0.0,
#                      "pole_velocity" : 0.02,
#                      "cart_velocity" : 0.1}
#             print(state)
#             action = self.choose_action(state)
#             direction = "왼쪽" if action == -1 else "오른쪽"
#             print(f"선택한 행동 {direction}")
#             reward = self.calculate_reward(state, action)
#             print(f"받은 보상 {reward}")
#             self.update_policy(state, action, reward)
#             print("전략 업데이트")
#             self.score += reward
#             if self.score < -100:
#                 self.game_over = True
#         print(f"최종 스코어 {self.score}")

#     def choose_action(self, state):
#         import random

#         if random.random() < 0.9:
#             return 1 if state["pole_angle"] > 0 else -1
#         else:
#             return random.choice([-1, 1])

#     def calculate_reward(self, state, action):
#         if abs(state["pole_angle"]) > 0.5:
#             return -100
#         else:
#             return +1

#     def update_policy(self, state, action, reward):
#         learning_rate = 0.1
#         discount_factor = 0.95
#         print('더 나은 전략으로 업데이트')

# demo = ReinforcementLearningDemo()
# demo.cartpole_example()

In [7]:
import gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque

class QNetWork(nn.Module):
    def __init__(self, state_size, action_size, seed, fc1_units = 64, fc2_units = 64):
        super(QNetWork, self).__init__()
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, fc1_units)
        self.fc2 = nn.Linear(fc1_units, fc2_units)
        self.fc3 = nn.Linear(fc2_units, action_size)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
seed = 42
qnetwork_local = QNetWork(state_size, action_size, seed)
qnetwork_target = QNetWork(state_size, action_size, seed)
optimizer = optim.Adam(qnetwork_local.parameters(), lr = 5e-4)

buffer_size = int(1e5)
batch_size = 64
memory = deque(maxlen = buffer_size)

def step(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

def sample():
    experiences = random.sample(memory, k = batch_size)
    states = torch.from_numpy(np.vstack([e[0] for e in experiences])).float()
    actions = torch.from_numpy(np.vstack([e[1] for e in experiences])).long()
    rewards = torch.from_numpy(np.vstack([e[2] for e in experiences])).float()
    next_states = torch.from_numpy(np.vstack([e[3] for e in experiences])).float()
    dones = torch.from_numpy(np.vstack([e[4] for e in experiences]).astype(np.uint8)).float()
    return (states, actions, rewards, next_states, dones)

def learn(experiences, gamma):
    states, actions, rewards, next_states, dones = experiences
    Q_targets_next = qnetwork_target(next_states).detach().max(1)[0].unsqueeze(1)
    Q_targets = rewards + (gamma * Q_targets_next * (1 - dones))
    Q_expected = qnetwork_local(states).gather(1, actions)
    loss = nn.MSELoss()(Q_expected, Q_targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

gamma = 0.99
tau = 1e-3
n_episodes = 200
max_t = 1000

for i_episode in range(1, n_episodes + 1):
    state = env.reset()
    total_reward = 0
    for t in range(max_t):
        state_tensor = torch.from_numpy(state).float().unsqueeze(0)
        with torch.no_grad():
            action_values = qnetwork_local(state_tensor)
        action = np.argmax(action_values.cpu().data.numpy())
        next_state, reward, done, _ = env.step(action)
        step(state, action, reward, next_state, done)
        total_reward += reward
        if len(memory) > batch_size:
            experiences = sample()
            loss = learn(experiences, gamma)
        state = next_state
        if done:
            break
    for target_param, local_param in zip(qnetwork_target.parameters(), qnetwork_local.parameters()):
        target_param.data.copy_(tau * local_param.data + (1 - tau) * target_param.data)
    print(i_episode, total_reward)

env.close()

1 9.0
2 10.0
3 10.0
4 9.0
5 10.0
6 10.0
7 10.0
8 9.0
9 10.0
10 8.0
11 9.0
12 10.0
13 9.0
14 9.0
15 9.0
16 9.0
17 10.0
18 9.0
19 9.0
20 10.0
21 9.0
22 10.0
23 9.0
24 10.0
25 10.0
26 8.0
27 9.0
28 9.0
29 9.0
30 10.0
31 9.0
32 9.0
33 10.0
34 9.0
35 10.0
36 10.0
37 11.0
38 9.0
39 9.0
40 10.0
41 10.0
42 9.0
43 8.0
44 10.0
45 9.0
46 10.0
47 9.0
48 10.0
49 9.0
50 10.0
51 9.0
52 9.0
53 10.0
54 10.0
55 10.0
56 9.0
57 9.0
58 10.0
59 9.0
60 9.0
61 9.0
62 8.0
63 9.0
64 9.0
65 9.0
66 9.0
67 10.0
68 9.0
69 10.0
70 9.0
71 8.0
72 9.0
73 8.0
74 9.0
75 9.0
76 9.0
77 9.0
78 11.0
79 10.0
80 10.0
81 11.0
82 9.0
83 9.0
84 9.0
85 8.0
86 9.0
87 10.0
88 9.0
89 8.0
90 11.0
91 9.0
92 10.0
93 9.0
94 9.0
95 11.0
96 10.0
97 9.0
98 8.0
99 10.0
100 9.0
101 9.0
102 10.0
103 10.0
104 10.0
105 10.0
106 11.0
107 9.0
108 10.0
109 8.0
110 9.0
111 9.0
112 10.0
113 10.0
114 10.0
115 9.0
116 10.0
117 10.0
118 10.0
119 10.0
120 10.0
121 10.0
122 9.0
123 9.0
124 11.0
125 10.0
126 8.0
127 9.0
128 11.0
129 10.0
130 10.0
131 9.0
1

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym

class PolicyNetWork(nn.Module):
    def __init__(self, state_size, action_size, hidden_dim = 128):
        super(PolicyNetWork, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, action_size)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = self.fc2(x)
        return torch.softmax(x, dim = -1)

env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

policy_net = PolicyNetWork(state_size, action_size)
optimizer = optim.Adam(policy_net.parameters(), lr = 0.01)

def select_action(state):
    state = torch.from_numpy(state).float().unsqueeze(0)
    probs = policy_net(state)
    action = torch.multinomial(probs, num_samples = 1)
    return action.item(), torch.log(probs[0, action.item()])

def reinforce_update(episode_rewards, episode_log_probs, gamma = 0.99):
    R = 0
    returns = []
    for r in episode_rewards[::-1]:
        R = r + gamma * R
        returns.insert(0, R)
    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    loss = 0
    for log_prob, R in zip(episode_log_probs, returns):
        loss -= log_prob * R
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

num_episodes = 1000
for episode in range(num_episodes):
    state = env.reset()
    episode_rewards = []
    episode_log_probs = []
    done = False
    while not done:
        action, log_prob = select_action(state)
        next_state, reward, done, _ = env.step(action)
        episode_rewards.append(reward)
        episode_log_probs.append(log_prob)
        state = next_state
    loss = reinforce_update(episode_rewards, episode_log_probs) 
    if episode % 50 == 0:
        total_reward = sum(episode_rewards)
        print(episode, total_reward, loss)

env.close()


0 18.0 -0.12988460063934326
50 177.0 1.6348743438720703
100 500.0 1.9380006790161133
150 500.0 -15.330184936523438
200 45.0 -2.2751126289367676
250 121.0 6.068619728088379
300 129.0 -4.663674354553223
350 264.0 18.085086822509766
400 249.0 3.332719326019287
450 500.0 3.753337860107422
500 240.0 -3.675839424133301
550 500.0 -4.127986907958984
600 500.0 3.5104310512542725
650 500.0 3.119297981262207
700 500.0 -6.505832672119141
750 500.0 -15.413211822509766
800 500.0 -0.10409511625766754
850 500.0 0.1376371681690216
900 500.0 14.656451225280762
950 500.0 2.133819580078125
